# 深度因子研究

本项目用日频价量因子训练横截面排序模型，将模型分数及其滚动分位数转换为每日组合权重，通过十分层回测观察选股信号的收益梯度、换手成本和回撤。

数据流：**日行情 → Alpha 因子 → 20 日分位数特征 → 神经网络分数 → 5/10/20 日滚动排名 → 十分层组合**。训练标签用于监督学习；推理和回测不读取未来标签。

各计算 Notebook 按「导入与参数 → 预加载 → 主循环 → 结果展示」组织。日期和文件地址一次整理，行情按日读取，历史窗口由有界队列维护。详细使用方法、数据字段和实验结果见 [README](README.md)。

## 研究步骤与实验配置

| 入口 | 工作内容 | 数据或结果目录 |
| --- | --- | --- |
| 01_原始因子 | 根据行情计算 101 列 Alpha 因子 | output |
| 02_模型输入 | 构造 45 维、20 日滚动分位数特征 | rank_output/factor_rank_20日 |
| train.py | 使用 2020–2022 年特征与标签训练排序模型 | 默认保存 PAIR |
| 03_模型因子 | 加载模型，生成分数及 5/10/20 日滚动排名 | 当前为 model2_output |
| 04_十分层回测 | 评估 PAIR（10 epoch）的四种信号 | model_output/backtest |
| 05_十分层回测50epoch | 评估 PAIR50E（50 epoch）的四种信号 | model2_output/backtest |

两个模型使用相同网络结构和训练数据，实验设置的区别是训练轮数。当前 03 的参数单元选择 `PAIR50E`；评估 `PAIR` 时应同时选择该模型和 `model_output` 输出目录。各输出目录中的 `PAIR`、`PAIR_rank` 是统一的数据子目录名，不表示训练轮数。

已有输入可直接用于训练和推理，无需重复计算 01、02。模型输出包含 2022 年末 19 个预热交易日；正式回测覆盖 2023-01-03 至 2026-09-30，共 908 天，跨年连续持仓。

## 时间对齐与计算口径

- 股票池为当日在市普通股（CS），排除北交所。01 的行情队列长度为 25 日；输出保留 101 列，历史长度不足、缺少行业或市值数据的因子保留 NaN。当日成交量非正时，因子置为 NaN。
- 02 从 `ALPHA_LIST` 选取固定顺序的 45 列。各日先进行横截面分位数排名，再对最近 20 日等权平均，最后再次横截面排名；均值跳过 NaN，并列排名取平均。
- 日期 T 的模型输入在 T 日原始因子入队前生成，使用 T−20 至 T−1 的信息。缓存保留 30 日，其股票并集决定输入表的资产索引。
- 模型输入缺失值填 0.5，使用 float32。模型在 `eval()` 和 `inference_mode()` 下输出每只股票的一个分数。
- 模型 T 日分数已在 T 日开盘前可用，因此模型分数的 5/10/20 日平滑包含 T。窗口未满时相应输出为 NaN。
- 回测先按完整信号截面的分位数区间 `(g/10, (g+1)/10]` 划组，再筛选当日在市股票，组内等权。并列分数不拆分，各组人数可能不同。
- 组合每天开盘调仓，单边费率为 0.00025；旧仓从行情中消失时按零残值计账。收益使用相邻两日的 close/open 与 open/prev_close 比例连接。
- 最后记录为末日开盘调仓后的净值，不含末日后续收益。成交模型没有额外的涨跌停和停牌成交限制。

2022 年末的训练标签包含部分 2023 年行情，因此回测起始段存在训练标签窗口重叠；解释样本外表现时需区分这一边界。

## 模型训练

在项目目录运行 `python train.py`。`FEATURE_PATH`、`LABEL_PATH`、`TRAIN`、`SEED`、`MODEL_PATH` 和 `num_epochs` 控制输入、训练年份、随机种子、保存位置和训练轮数。默认训练 10 轮并保存到 `PAIR`；50 轮实验应将 `num_epochs` 设为 50，并将 `MODEL_PATH` 指向 `PAIR50E`。

每个训练样本是一整个交易日的股票截面，DataLoader 仅打乱日期顺序。X、Y 按同日及股票代码对齐：X 的缺失值填 0.5；标签缺失值填 0 后，对未来十段日收益分别做横截面分位数排名，再对十列取均值得到 Y。

网络为 45 维输入、256 维隐藏层、4 个残差块和单值输出，使用 GELU 与 0.1 Dropout。训练采用全部股票对的 Softplus 排序损失与预测值平方正则项（alpha=0.01），优化器为 Adam，学习率 0.001，随机种子 42。

每轮打印各日 Pearson IC 的均值、方差和未年化 ICIR。这些统计来自训练模式下不断更新的模型，目标是十段收益的平均排名，不等同于最终固定模型的样本外收益评估。相同种子也不保证 CPU 与 GPU 的参数逐位一致。

## 工具与数据

- `TOOL/data.py`：路径、交易日期预加载、股票池筛选、逐日文件读写。
- `TOOL/Struct.py`：行情队列与时间 × 资产 × 字段的数据容器。
- `TOOL/OperatorHybrid.py`、`TOOL/WQalpha.py`、`TOOL/factor.py`：数值算子、Alpha 公式、因子计算与滚动排名。
- `TOOL/model.py`：按日数据集、残差网络、排序损失及模型加载。
- `TOOL/book.py`、`TOOL/backtest.py`、`TOOL/show_book.py`：组合记账、结果汇总和可视化。
- `codex/`：开发辅助脚本及检查记录，不是日常研究的入口。

模型、训练特征和标签均位于本目录。重新计算原始因子或执行回测时，需要 `TOOL/data.py` 中 `DAILY_DATA_PATH` 指向的行情目录；默认地址是 `/mnt/nas/data/whole_history`。更换工作目录时也应检查 03、05 的实验参数单元中的路径。